# EDGE — 90s Talking-Head Explainer (Google Colab / SadTalker)

**Before running:** Runtime → Change runtime type → select **GPU** → Save.
Then Run All. This generates a real lip-synced talking-head video from ONE photo + script.

Outputs: `edge_company_intro_90sec.mp4`, `.srt`, `_script.txt`, `README.md`


In [ ]:
# 0. Install SadTalker + deps (Colab has GPU + internet)
import os, subprocess, sys
from IPython.display import clear_output
if not os.path.exists('/content/SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker
os.chdir('/content/SadTalker')
# Install binary + python deps via requirements (Colab python is compatible)
!pip install -q imageio-ffmpeg  # provides ffmpeg binary
for pkg in ['kornia==0.6.8','face_alignment==1.3.5','basicsr==1.4.2','facexlib==0.3.0','gfpgan','av','safetensors','einops','opencv-python-headlib','opencv-python-headless']:
    subprocess.run([sys.executable,'-m','pip','install','-q',pkg],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
print('deps attempted')


In [ ]:
# 1. Download model checkpoints (one-time)
os.chdir('/content/SadTalker')
!bash scripts/download_models.sh
print('models present' if os.path.exists('checkpoints/SadTalker_V0.0.2_512.safetensors') else 'MODEL DOWNLOAD FAILED')


In [ ]:
# 2. Fetch profile image (exact face reference)
import urllib.request, os
IMG_ID='1-2sFUEHqXDbaPq0lfBmamrjQBsdL_QuY'
os.makedirs('/content/assets',exist_ok=True)
urllib.request.urlretrieve(f'https://drive.google.com/uc?export=download&id={IMG_ID}','/content/assets/profile.jpg')
print('profile.jpg', os.path.getsize('/content/assets/profile.jpg'),'bytes')


In [ ]:
# 3. Script (verbatim) + male voiceover (free TTS, 3x30s)
import urllib.request, urllib.parse, subprocess, os
SCRIPT = '''
I'm Mirsina Aghdam, CEO of EDGE - Earthwise Dynamics Geo Environs. We're an Irish research-driven startup operating at the intersection of geoengineering, AI and space science.

Europe imports almost all the rare earth elements it needs for electric vehicles, wind turbines and defence systems. Current exploration methods are slow, expensive and miss low-grade deposits that could be strategic assets.

EDGE has developed TerraLens - an AI-enabled platform that combines drone-mounted gamma spectroscopy, LiDAR, satellite data and machine learning to locate and quantify rare earth mineralisation. We're funded by ESA Business Incubation Centre Ireland to build this technology.

Phase one is terrestrial - we're validating TerraLens through field surveys across European sites, proving the technology works on real geology. Phase two builds on that foundation to create a space-based version for lunar and asteroid prospecting. Russia and China are already moving on lunar resources. Europe needs this capability.

We're a team of senior engineers dedicated to pushing boundaries in geospatial modelling and automated AI solutions. If you're working on critical minerals, space resources or AI-driven exploration, let's talk.
'''
with open('/content/assets/script.txt','w') as f: f.write(SCRIPT)
words=SCRIPT.replace('\n',' ').split(); chunks=[]; cur=''
for w in words:
    if len(cur)+len(w)>150: chunks.append(cur); cur=''
    cur+=(' ' if cur else '')+w
if cur: chunks.append(cur)
for i,p in enumerate(chunks):
    u='https://translate.google.com/translate_tts?ie=UTF-8&client=tw-ob&tl=en-GB&q='+urllib.parse.quote(p)
    urllib.request.urlretrieve(u,f'/content/assets/tts_{i}.mp3')
open('/content/assets/tts_list.txt','w').write('\n'.join(f"file '/content/assets/tts_{i}.mp3'" for i in range(len(chunks))))
FF=subprocess.check_output(['python','-c','import imageio_ffmpeg,os;print(imageio_ffmpeg.get_ffmpeg_exe())']).decode().strip()
subprocess.run([FF,'-y','-f','concat','-safe','0','-i','/content/assets/tts_list.txt','-ar','16000','-ac','1','/content/assets/voice.wav'],check=True)
for i in range(3):
    subprocess.run([FF,'-y','-i','/content/assets/voice.wav','-ss',str(i*30),'-t',('31' if i==2 else '30'),'-ar','16000','-ac','1',f'/content/assets/seg_{i}.wav'],check=True)
print('voice + segments ready')


In [ ]:
# 4. Generate talking head with SadTalker (one segment at a time)
import subprocess, os
os.chdir('/content/SadTalker')
FF=subprocess.check_output(['python','-c','import imageio_ffmpeg;print(imageio_ffmpeg.get_ffmpeg_exe())']).decode().strip()
for i in range(3):
    cmd=['python','inference.py','--driven_audio',f'/content/assets/seg_{i}.wav','--source_image','/content/assets/profile.jpg','--result_dir','/content/results','--size','512','--preprocess','crop','--enhancer','gfpgan','--batch_size','1']
    print(f'--- segment {i} ---'); subprocess.run(cmd,check=False)
print('sadtalker done')


In [ ]:
# 5. Stitch + upscale to 1080p
import glob, os, subprocess
FF=subprocess.check_output(['python','-c','import imageio_ffmpeg;print(imageio_ffmpeg.get_ffmpeg_exe())']).decode().strip()
os.makedirs('/content/final',exist_ok=True)
segs=sorted(glob.glob('/content/results/*/*.mp4')); print('segments:',segs)
with open('/content/final/concat.txt','w') as f:
    for s in segs: f.write(f"file '{s}'\n")
subprocess.run([FF,'-y','-f','concat','-safe','0','-i','/content/final/concat.txt','-c','copy','/content/final/stitched.mp4'],check=True)
subprocess.run([FF,'-y','-i','/content/final/stitched.mp4','-vf','scale=1920:1080:flags=lanczos','-r','30','-c:v','libx264','-crf','20','-pix_fmt','yuv420p','-ar','48000','-c:a','aac','-b:a','160k','/content/final/edge_company_intro_90sec.mp4'],check=True)
print('final ready')


In [ ]:
# 6. Burn-in subtitles + 4 title cards
import subprocess, os
FF=subprocess.check_output(['python','-c','import imageio_ffmpeg;print(imageio_ffmpeg.get_ffmpeg_exe())']).decode().strip()
OUT='/content/final/edge_company_intro_90sec.mp4'; TMP='/content/final/subtitled.mp4'
open('/content/final/subs.srt','w').write('''1\n00:00:00,000 --> 00:00:14,000\nI'm Mirsina Aghdam, CEO of EDGE - Earthwise Dynamics Geo Environs.\n\n2\n00:00:14,000 --> 00:00:30,000\nEurope imports almost all the rare earth elements it needs.\n\n3\n00:00:30,000 --> 00:00:50,000\nEDGE developed TerraLens: drone gamma spectroscopy, LiDAR, satellite, AI.\n\n4\n00:00:50,000 --> 00:01:15,000\nPhase 1 terrestrial. Phase 2 space-based lunar and asteroid prospecting.\n\n5\n00:01:15,000 --> 00:01:30,000\nSenior engineers building better decisions on Europe's mineral resources. Let's talk.\n''')
cards=["drawtext=text='Mirsina Aghdam | CEO, EDGE':fontcolor=white:fontsize=42:x=(w-tw)/2:y=h-120:enable='between(t,1,12)'",
       "drawtext=text='TerraLens AI | Rare Earth Exploration Intelligence':fontcolor=white:fontsize=38:x=(w-tw)/2:y=h-120:enable='between(t,30,48)'",
       "drawtext=text='Phase 1 | Terrestrial validation':fontcolor=white:fontsize=38:x=(w-tw)/2:y=h-120:enable='between(t,50,66)'",
       "drawtext=text='Phase 2 | Space-enabled mineral intelligence':fontcolor=white:fontsize=36:x=(w-tw)/2:y=h-120:enable='between(t,66,82)'"]
vf=",".join(["subtitles='/content/final/subs.srt':force_style='FontSize=22,PrimaryColour=&H00FFFFFF,BackColour=&H80000000'"]+cards)
subprocess.run([FF,'-y','-i',OUT,'-vf',vf,'-c:v','libx264','-crf','20','-c:a','copy',TMP],check=True)
os.replace(TMP,OUT); print('subtitles + cards burned in')


In [ ]:
# 7. Collect deliverables + download links
import shutil, os
shutil.copy('/content/final/edge_company_intro_90sec.mp4','/content/edge_company_intro_90sec.mp4')
shutil.copy('/content/assets/script.txt','/content/edge_company_intro_90sec_script.txt')
shutil.copy('/content/final/subs.srt','/content/edge_company_intro_90sec.srt')
open('/content/README.md','w').write('EDGE 90s Talking-Head\nModel: SadTalker on Colab GPU\n1920x1080, 30fps, H.264, AAC\nSegments 3x30s stitched + upscaled + subtitled.\n')
from google.colab import files
for f in ['edge_company_intro_90sec.mp4','edge_company_intro_90sec.srt','edge_company_intro_90sec_script.txt','README.md']:
    if os.path.exists('/content/'+f): files.download('/content/'+f)
print('done - files downloading')
